In [68]:
import sys
#%load_ext autoreload
#%autoreload 2

# for data collection
from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD_lab12
import time

# for plotting
import numpy as np
import matplotlib.pyplot as plt
from scipy.fftpack import fft

# for calculation
import math

LOG.propagate = False

ble = get_ble_controller()
ble.connect()

2026-05-11 13:38:05,046 | INFO     |: Looking for Artemis Nano Peripheral Device: c0:81:b4:24:2b:64
2026-05-11 13:38:05,051 | INFO     |: Scanning for device with address: c0:81:b4:24:2b:64, service UUID: 15bb5de7-5941-4ba2-bda0-784bb8817a1b
2026-05-11 13:38:15,140 | INFO     |: Found 1 device(s) advertising service 15bb5de7-5941-4ba2-bda0-784bb8817a1b
2026-05-11 13:38:15,148 | INFO     |: Selecting device: 38645F53-E5BF-2155-7DF0-DBDE5B0B8B54 (name: Artemis BLE)
2026-05-11 13:38:15,995 | INFO     |: Connected to c0:81:b4:24:2b:64


In [69]:
# notification handler so GET_YAW return data
# for debugging, to read current yaw from robot
latest_yaw = [None]

def yaw_notif_handler(uuid, byte_array):
    try:
        s = ble.bytearray_to_string(byte_array)
        # firmware sends "Y:<float>"
        if s.startswith("Y:"):
            latest_yaw[0] = float(s[2:])
    except Exception as e:
        print(f"[notif] error: {e} | raw: {s}")
        
ble.start_notify(ble.uuid['RX_STRING'], yaw_notif_handler)

In [56]:
# calibrate forward speed
# drive forward at selected PWm for 1 second, then measure how far the robot traveled
PWM_DRIVE = 120 # open loop, lower PWM is better... but higher than 100 
ble.send_command(CMD_lab12.DRIVE_OPEN_LOOP, f"{PWM_DRIVE}|1000")

In [74]:
# calculated forward speed
SPEED_FT_PER_S = 5.0

In [71]:
# calibrate turn
# rotate 90 degree CCW
ble.send_command(CMD_lab12.RESET_YAW, "")
time.sleep(0.2)
ble.send_command(CMD_lab12.START_PID, "")
time.sleep(0.2)
ble.send_command(CMD_lab12.SET_ORIENTATION_SETPOINT, "90.0")
time.sleep(3.0)
ble.send_command(CMD_lab12.STOP_PID, "")

# read actual yaw to check accuracy
ble.send_command(CMD_lab12.GET_YAW, "")
time.sleep(0.3)
print(f"Target = 90 deg, actual yaw = {latest_yaw[0]}")

Target = 90 deg, actual yaw = None


In [75]:
# waypoints + planner
# all waypoints in feet, world coords
# World +x = east
# World +y = north
# currently robot place at waypoint 0, facing +x

WAYPOINTS_FT = [(-4, -3), (-2, -1), (1, -1), (2, -3), (5, -3), (5, -2), (5,  3), (0,  3), (0,  0),]

# for debugging and tuning
PWM_DRIVE = 120
TURN_SETTLE_S = 2.5 # turn settle time
TURN_PER_DEG = 0.015 # extra settle time per degree of turn
DRIVE_BUFFER_S = 0.4 # post-drive wait so robot fully stops
INTER_LEG_S = 0.5 # time between legs

# Wrap angle to (-180, 180]
def normalize_angle(a):
    while a >  180.0: a -= 360.0
    while a <= -180.0: a += 360.0
    return a

# Return (target_heading_world_deg, delta_theta_deg, distance_ft)
def plan_leg(cur_pose, next_wp_ft):
    cur_x, cur_y, cur_theta = cur_pose
    nx, ny = next_wp_ft
    dx = nx - cur_x
    dy = ny - cur_y
    desired = math.degrees(math.atan2(dy, dx))    # world heading needed
    delta   = normalize_angle(desired - cur_theta)
    dist    = math.hypot(dx, dy)
    return desired, delta, dist
    
# Turn-then-drive. Updates and returns the new pose
def execute_leg(cur_pose, next_wp_ft, leg_idx):
    desired, delta, dist_ft = plan_leg(cur_pose, next_wp_ft)
    drive_time_s  = dist_ft / SPEED_FT_PER_S
    drive_time_ms = int(drive_time_s * 1000)
    turn_wait_s   = TURN_SETTLE_S + abs(delta) * TURN_PER_DEG
 
    print(f"\nLeg {leg_idx}: {cur_pose[:2]} -> {next_wp_ft}")
    print(f"  delta_theta={delta:+.1f}° (target world heading {desired:+.1f}°), "f"dist={dist_ft:.2f} ft -> drive {drive_time_ms} ms")
 
    # 1: Turn (firmware uses absolute setpoint in world frame)
    if abs(delta) > 1.0:
        ble.send_command(CMD_lab12.SET_ORIENTATION_SETPOINT, f"{desired:.2f}")
        time.sleep(turn_wait_s)
    else:
        print("  (no turn needed)")
 
    # 2: Drive open-loop
    ble.send_command(CMD_lab12.DRIVE_OPEN_LOOP, f"{PWM_DRIVE}|{drive_time_ms}")
    time.sleep(drive_time_s + DRIVE_BUFFER_S)
 
    # 3: Update expected pose. Assume we hit the waypoint and ended at desired heading.
    new_pose = (next_wp_ft[0], next_wp_ft[1], desired)
    print(f"  expected pose: ({new_pose[0]:.1f}, {new_pose[1]:.1f}, {new_pose[2]:+.1f}°)")
 
    time.sleep(INTER_LEG_S)
    return new_pose

In [76]:
# run full trajectory
# reset everything
ble.send_command(CMD_lab12.RESET_YAW, "")
time.sleep(0.3)
ble.send_command(CMD_lab12.START_PID, "")
time.sleep(0.3)
# initial pose at waypoint 0, heading 0°, facing +x
pose = (WAYPOINTS_FT[0][0], WAYPOINTS_FT[0][1], 0.0)
try:
    for i in range(1, len(WAYPOINTS_FT)):
        pose = execute_leg(pose, WAYPOINTS_FT[i], i)
    print("\nTrajectory complete!")
finally:
    ble.send_command(CMD_lab12.STOP_PID, "")
    print("PID stopped, motors turn off.")


Leg 1: (-4, -3) -> (-2, -1)
  delta_theta=+45.0° (target world heading +45.0°), dist=2.83 ft -> drive 565 ms
  expected pose: (-2.0, -1.0, +45.0°)

Leg 2: (-2, -1) -> (1, -1)
  delta_theta=-45.0° (target world heading +0.0°), dist=3.00 ft -> drive 600 ms
  expected pose: (1.0, -1.0, +0.0°)

Leg 3: (1, -1) -> (2, -3)
  delta_theta=-63.4° (target world heading -63.4°), dist=2.24 ft -> drive 447 ms
  expected pose: (2.0, -3.0, -63.4°)

Leg 4: (2, -3) -> (5, -3)
  delta_theta=+63.4° (target world heading +0.0°), dist=3.00 ft -> drive 600 ms
  expected pose: (5.0, -3.0, +0.0°)

Leg 5: (5, -3) -> (5, -2)
  delta_theta=+90.0° (target world heading +90.0°), dist=1.00 ft -> drive 200 ms
  expected pose: (5.0, -2.0, +90.0°)

Leg 6: (5, -2) -> (5, 3)
  delta_theta=+0.0° (target world heading +90.0°), dist=5.00 ft -> drive 1000 ms
  (no turn needed)
  expected pose: (5.0, 3.0, +90.0°)

Leg 7: (5, 3) -> (0, 3)
  delta_theta=+90.0° (target world heading +180.0°), dist=5.00 ft -> drive 1000 ms
  exp

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x108f8cac0> is already entered



Leg 8: (0, 3) -> (0, 0)
  delta_theta=+90.0° (target world heading -90.0°), dist=3.00 ft -> drive 600 ms
  expected pose: (0.0, 0.0, -90.0°)

Trajectory complete!
PID stopped, motors turn off.


In [26]:
# to debug individual leg
# place robot at WAYPOINTS_FT[start_wp_idx] facing start_heading_deg
# then drive a single leg to WAYPOINTS_FT[end_wp_idx]
def run_one_leg(start_wp_idx, end_wp_idx, start_heading_deg=0.0):
    ble.send_command(CMD_lab12.RESET_YAW, "")
    time.sleep(0.2)
    ble.send_command(CMD_lab12.START_PID, "")
    time.sleep(0.2)
    pose = (WAYPOINTS_FT[start_wp_idx][0],
            WAYPOINTS_FT[start_wp_idx][1],
            start_heading_deg)
    pose = execute_leg(pose, WAYPOINTS_FT[end_wp_idx], end_wp_idx)
    ble.send_command(CMD_lab12.STOP_PID, "")
    return pose

# to test --> for example, leg 0 to 1: run_one_leg(0, 1)
    

In [34]:
run_one_leg(0, 1, start_heading_deg=0.0)


Leg 1: (-4, -3) -> (-2, -1)
  delta_theta=+45.0° (target world heading +45.0°), dist=2.83 ft -> drive 514 ms
  expected pose: (-2.0, -1.0, +45.0°)


(-2, -1, 45.0)

In [35]:
run_one_leg(1, 2, start_heading_deg=0.0)


Leg 2: (-2, -1) -> (1, -1)
  delta_theta=+0.0° (target world heading +0.0°), dist=3.00 ft -> drive 545 ms
  (no turn needed)
  expected pose: (1.0, -1.0, +0.0°)


(1, -1, 0.0)

In [ ]:
# stop robot
ble.send_command(CMD_lab12.STOP_PID, "")